# 01 · Dataset

Genera los ejemplos sintéticos, valida todo el dataset y arma los splits estratificados por `prioridad`. Los ejemplos `manual`/`real` los escribe el equipo a mano.

In [ ]:
# === Setup (en Colab se ejecuta solo; en local no hace nada) ===
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    !pip install -q -r requirements.txt
    !pip install -q -e ".[gpu]"

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, las salidas van a Drive para sobrevivir desconexiones.
if EN_COLAB:
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir  = config.drive_dir / "adaptadores"
    config.results_dir.mkdir(parents=True, exist_ok=True)
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name, "| resultados en:", config.results_dir)

## Generar ejemplos sintéticos

Solo `origen: sintetico`. El 20% manual lo escribe el equipo.

In [ ]:
!python scripts/generar_sintetico.py --n 1200 --seed 42 --salida data/sintetico/lote01.jsonl

## Validar todo el dataset (manual + real + sintetico)

In [ ]:
!python scripts/validar_dataset.py data/sintetico/*.jsonl data/manual/*.jsonl

## Construir y guardar los splits (estratificados, semilla fija)

In [ ]:
from triaje.data import cargar_todos, dividir, guardar_splits
ejemplos = cargar_todos(config.data_dir)
splits = dividir(ejemplos, config.frac_train, config.frac_val, config.frac_test, seed=config.seed_split)
print(splits.resumen())
guardar_splits(splits, config.data_dir / "splits")